# BasementDrugDiscovery
## Notebook 04 -- Molecular Docking

**What this notebook does:**
Screens all prepared FDA approved ligands against each CYP51 receptor using AutoDock Vina.
For each docking run it records the binding affinity, the interacting residues, and all relevant
docking data directly into the existing SQLite database -- one record per ligand per target.
A checkpoint system prevents duplicate runs if the process is interrupted.

**Input:**
- Receptor PDBQT files and docking_targets.json from Notebook 03
- Ligand PDBQT files and bdd_molecules.db from Notebook 02

**Output:**
- docking_results table added to bdd_molecules.db
- Best pose PDBQT files saved per target

---

### Cell 1 -- Load all required tools

In [1]:
import os
import json
import sqlite3
import subprocess
import tkinter as tk
from tkinter import filedialog, simpledialog
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

from Bio.PDB import PDBParser
from vina import Vina

import ipywidgets as widgets
from IPython.display import display, clear_output

def get_tk_root():
    root = tk.Tk()
    root.withdraw()
    root.attributes('-topmost', True)
    return root

def browse_file(title='Select file', filetypes=None):
    if filetypes is None:
        filetypes = [('All files', '*.*')]
    root = get_tk_root()
    path = filedialog.askopenfilename(title=title, filetypes=filetypes)
    root.destroy()
    return path

def browse_directory(title='Select folder'):
    root = get_tk_root()
    path = filedialog.askdirectory(title=title)
    root.destroy()
    return path

def ask_string(title, prompt, initial=''):
    root = get_tk_root()
    result = simpledialog.askstring(title=title, prompt=prompt, initialvalue=initial, parent=root)
    root.destroy()
    return result

def ask_integer(title, prompt, initial=1):
    root = get_tk_root()
    result = simpledialog.askinteger(title=title, prompt=prompt, initialvalue=initial, parent=root)
    root.destroy()
    return result

def ask_float(title, prompt, initial=0.0):
    root = get_tk_root()
    result = simpledialog.askfloat(title=title, prompt=prompt, initialvalue=initial, parent=root)
    root.destroy()
    return result

print('All tools loaded.')
print('Popup file browsers ready.')

All tools loaded.
Popup file browsers ready.


### Cell 2 -- Select input files

Click each button to select the required files. No typing required.

In [2]:
selected = {
    'db_path': None,
    'ligand_dir': None,
    'receptor_pdbqt': None,
    'poses_dir': None,
    'targets_json': None
}

docking_config = {
    'target_name': None,
    'target_info': None,
    'exhaustiveness': 8,
    'n_poses': 3,
    'cpu': 4
}

status_out = widgets.Output()

btn_db = widgets.Button(description='Browse -- Select bdd_molecules.db',
    button_style='primary', layout=widgets.Layout(width='380px'))
btn_ligands = widgets.Button(description='Browse -- Select PDBQT ligand folder',
    button_style='primary', layout=widgets.Layout(width='380px'))
btn_receptor = widgets.Button(description='Browse -- Select receptor PDBQT file',
    button_style='primary', layout=widgets.Layout(width='380px'))
btn_poses = widgets.Button(description='Browse -- Select folder to save docking poses',
    button_style='primary', layout=widgets.Layout(width='380px'))
btn_targets_json = widgets.Button(description='Browse -- Select docking_targets.json',
    button_style='primary', layout=widgets.Layout(width='380px'))
btn_grid = widgets.Button(description='Enter target name and binding site coordinates',
    button_style='success', layout=widgets.Layout(width='380px'))

def sel_db(b):
    with status_out:
        p = browse_file('Select bdd_molecules.db',
            [('SQLite DB', '*.db'), ('All files', '*.*')])
        if p:
            selected['db_path'] = p
            print(f'Database: {p}')

def sel_ligands(b):
    with status_out:
        p = browse_directory('Select folder containing ligand PDBQT files')
        if p:
            count = len(list(Path(p).glob('*.pdbqt')))
            selected['ligand_dir'] = p
            print(f'Ligand folder: {p}')
            print(f'PDBQT files found: {count}')

def sel_receptor(b):
    with status_out:
        p = browse_file('Select receptor PDBQT file',
            [('PDBQT files', '*.pdbqt'), ('All files', '*.*')])
        if p:
            selected['receptor_pdbqt'] = p
            print(f'Receptor PDBQT: {p}')

def sel_poses(b):
    with status_out:
        p = browse_directory('Select folder to save docking poses')
        if p:
            selected['poses_dir'] = p
            print(f'Poses folder: {p}')

def sel_targets_json(b):
    with status_out:
        p = browse_file('Select docking_targets.json',
            [('JSON files', '*.json'), ('All files', '*.*')])
        if p:
            selected['targets_json'] = p
            with open(p) as f:
                config = json.load(f)
            print(f'Targets config: {p}')
            print(f'Targets available: {list(config.keys())}')

def enter_grid(b):
    with status_out:
        target_name = ask_string('Target Name',
            'Enter a short name for this target (e.g. candida_albicans):',
            'candida_albicans')
        pdb_id = ask_string('PDB ID',
            'Enter the PDB ID of this structure:', '5V5Z')
        cx = ask_float('Center X', 'Binding site center X (from PyMOL):', 0.0)
        cy = ask_float('Center Y', 'Binding site center Y (from PyMOL):', 0.0)
        cz = ask_float('Center Z', 'Binding site center Z (from PyMOL):', 0.0)
        sx = ask_float('Size X', 'Docking box size X in Angstroms:', 30.0)
        sy = ask_float('Size Y', 'Docking box size Y in Angstroms:', 30.0)
        sz = ask_float('Size Z', 'Docking box size Z in Angstroms:', 30.0)

        if None not in (target_name, cx, cy, cz, sx, sy, sz):
            docking_config['target_name'] = target_name.strip().lower().replace(' ', '_')
            docking_config['target_info'] = {
                'pdb_id': pdb_id.strip().upper() if pdb_id else 'UNKNOWN',
                'receptor_pdbqt': selected['receptor_pdbqt'],
                'grid_box': {
                    'center_x': cx, 'center_y': cy, 'center_z': cz,
                    'size_x': sx, 'size_y': sy, 'size_z': sz
                }
            }
            print(f'Target: {docking_config["target_name"]} ({pdb_id})')
            print(f'Grid center: ({cx}, {cy}, {cz})')
            print(f'Grid size: ({sx}, {sy}, {sz})')
            print('Ready. Run Cell 4 to initialize the database.')

btn_db.on_click(sel_db)
btn_ligands.on_click(sel_ligands)
btn_receptor.on_click(sel_receptor)
btn_poses.on_click(sel_poses)
btn_targets_json.on_click(sel_targets_json)
btn_grid.on_click(enter_grid)

print('Select all inputs in order:')
display(btn_db)
display(btn_ligands)
display(btn_receptor)
display(btn_poses)
display(btn_targets_json)
display(btn_grid)
display(status_out)

Select all inputs in order:


Button(button_style='primary', description='Browse -- Select bdd_molecules.db', layout=Layout(width='380px'), …

Button(button_style='primary', description='Browse -- Select PDBQT ligand folder', layout=Layout(width='380px'…

Button(button_style='primary', description='Browse -- Select receptor PDBQT file', layout=Layout(width='380px'…

Button(button_style='primary', description='Browse -- Select folder to save docking poses', layout=Layout(widt…

Button(button_style='primary', description='Browse -- Select docking_targets.json', layout=Layout(width='380px…

Button(button_style='success', description='Enter target name and binding site coordinates', layout=Layout(wid…

Output()

### Cell 3 -- Select target and configure docking parameters

Choose which target to dock against and set the number of poses to generate per ligand.

In [4]:
config_out = widgets.Output()

btn_target = widgets.Button(description='Select target to dock against',
    button_style='success', layout=widgets.Layout(width='350px'))
btn_params = widgets.Button(description='Set docking parameters',
    button_style='info', layout=widgets.Layout(width='350px'))

def select_target(b):
    with config_out:
        clear_output()
        if not selected['targets_json']:
            print('Select docking_targets.json first in Cell 2.')
            return
        with open(selected['targets_json']) as f:
            config = json.load(f)
        targets = list(config.keys())
        target = ask_string(
            'Select Target',
            f'Enter target name. Available: {targets}',
            initial=targets[0] if targets else ''
        )
        if target and target in config:
            docking_config['target_name'] = target
            docking_config['target_info'] = config[target]
            grid = config[target]['grid_box']
            print(f'Target selected: {target}')
            print(f'PDB ID: {config[target]["pdb_id"]}')
            print(f'Receptor PDBQT: {config[target]["receptor_pdbqt"]}')
            print(f'Grid center: ({grid["center_x"]}, {grid["center_y"]}, {grid["center_z"]})')
            print(f'Grid size: ({grid["size_x"]}, {grid["size_y"]}, {grid["size_z"]})')
        else:
            print(f'Target not found. Available: {targets}')

def set_params(b):
    with config_out:
        exhaustiveness = ask_integer('Exhaustiveness',
            'Docking exhaustiveness (8=standard, 16=thorough, 32=very thorough):', 8)
        n_poses = ask_integer('Number of Poses',
            'Number of poses to generate per ligand (1-9):', 3)
        cpu = ask_integer('CPU cores',
            'Number of CPU cores to use:', 4)
        if exhaustiveness: docking_config['exhaustiveness'] = exhaustiveness
        if n_poses: docking_config['n_poses'] = n_poses
        if cpu: docking_config['cpu'] = cpu
        print(f'Parameters set:')
        print(f'  Exhaustiveness: {docking_config["exhaustiveness"]}')
        print(f'  Poses per ligand: {docking_config["n_poses"]}')
        print(f'  CPU cores: {docking_config["cpu"]}')

btn_target.on_click(select_target)
btn_params.on_click(set_params)

display(btn_target)
display(btn_params)
display(config_out)

Button(button_style='success', description='Select target to dock against', layout=Layout(width='350px'), styl…

Button(button_style='info', description='Set docking parameters', layout=Layout(width='350px'), style=ButtonSt…

Output()

### Cell 4 -- Initialize database tables for docking results

Adds a docking_results table to the existing bdd_molecules.db if it does not already exist.
Also adds a docking_contacts table for residue contact information.
Safe to run multiple times -- existing data is never overwritten.

In [6]:
def initialize_docking_tables(db_path):
    """
    Add docking result tables to the existing database.
    Creates tables only if they do not already exist.
    """
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    # Main docking results -- one row per ligand per target
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS docking_results (
            result_id           INTEGER PRIMARY KEY AUTOINCREMENT,
            chembl_id           TEXT,
            target_name         TEXT,
            pdb_id              TEXT,
            affinity_best       REAL,
            affinity_2nd        REAL,
            affinity_3rd        REAL,
            n_poses             INTEGER,
            pose_file           TEXT,
            exhaustiveness      INTEGER,
            grid_center_x       REAL,
            grid_center_y       REAL,
            grid_center_z       REAL,
            grid_size_x         REAL,
            grid_size_y         REAL,
            grid_size_z         REAL,
            status              TEXT DEFAULT 'success',
            error_message       TEXT,
            timestamp           TEXT,
            UNIQUE(chembl_id, target_name)
        )
    ''')
    
    # Residue contacts -- one row per contacting residue per docking result
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS docking_contacts (
            contact_id          INTEGER PRIMARY KEY AUTOINCREMENT,
            result_id           INTEGER,
            chembl_id           TEXT,
            target_name         TEXT,
            residue_name        TEXT,
            residue_number      INTEGER,
            chain               TEXT,
            contact_distance    REAL,
            FOREIGN KEY (result_id) REFERENCES docking_results(result_id)
        )
    ''')
    
    conn.commit()
    
    # Show current state
    existing = cursor.execute(
        'SELECT COUNT(*) FROM docking_results'
    ).fetchone()[0]
    
    molecules = cursor.execute(
        "SELECT COUNT(*) FROM molecules WHERE status = 'success'"
    ).fetchone()[0]
    
    conn.close()
    
    print(f'Database: {db_path}')
    print(f'Tables initialized: docking_results, docking_contacts')
    print(f'Existing docking results in database: {existing}')
    print(f'Molecules available for docking: {molecules}')


if selected['db_path']:
    initialize_docking_tables(selected['db_path'])
else:
    print('No database selected. Run Cell 2 first.')

Database: /home/sardism/BasementDrugDiscovery/data/bdd_fdamolecules.db
Tables initialized: docking_results, docking_contacts
Existing docking results in database: 0
Molecules available for docking: 2864


### Cell 5 -- Checkpoint function

Checks which molecules have already been docked against the selected target.
Returns only molecules that still need processing.
Safe to call after any interruption.

In [7]:
def get_pending_docking(db_path, target_name, ligand_dir):
    """
    Find molecules that have not yet been docked against this target.
    Cross-references the molecules table with docking_results.
    """
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    # All successfully prepared molecules
    all_ready = cursor.execute(
        'SELECT chembl_id, name FROM molecules WHERE status = "success"'
    ).fetchall()
    
    # Already docked against this target
    already_done = set(row[0] for row in cursor.execute(
        'SELECT chembl_id FROM docking_results WHERE target_name = ?',
        (target_name,)
    ).fetchall())
    
    conn.close()
    
    # Filter to only those with existing PDBQT files and not yet docked
    ligand_dir = Path(ligand_dir)
    pending = []
    missing_pdbqt = 0
    
    for chembl_id, name in all_ready:
        if chembl_id in already_done:
            continue
        pdbqt_file = ligand_dir / f'{chembl_id}.pdbqt'
        if pdbqt_file.exists():
            pending.append((chembl_id, name, str(pdbqt_file)))
        else:
            missing_pdbqt += 1
    
    print(f'Target: {target_name}')
    print(f'Total prepared molecules: {len(all_ready)}')
    print(f'Already docked against this target: {len(already_done)}')
    print(f'Missing PDBQT files: {missing_pdbqt}')
    print(f'Pending for this run: {len(pending)}')
    
    return pending


if all(selected[k] for k in ['db_path', 'ligand_dir']) and docking_config['target_name']:
    pending_ligands = get_pending_docking(
        selected['db_path'],
        docking_config['target_name'],
        selected['ligand_dir']
    )
    print(f'\nReady to start docking.')
else:
    print('Complete Cell 2 and Cell 3 first.')

Target: candida_albicans_sc5314
Total prepared molecules: 2864
Already docked against this target: 0
Missing PDBQT files: 0
Pending for this run: 2864

Ready to start docking.


### Cell 6 -- Define contact analysis function

After each docking run, this function parses the best pose and the receptor to identify which protein residues are within contact distance of the docked ligand. Contacts are defined as any heavy atom within 4.5 Angstroms.

In [8]:
CONTACT_DISTANCE = 4.5  # Angstroms
HYDROGEN_TYPES = {'H', 'HD', 'HS'}  # AutoDock atom types for hydrogen

def parse_pdbqt_coords(pdbqt_path):
    """
    Extract heavy atom coordinates from the first model in a PDBQT file.
    Returns a numpy array of (x, y, z) coordinates.
    """
    coords = []
    with open(pdbqt_path) as f:
        for line in f:
            if line.startswith(('ATOM', 'HETATM')):
                try:
                    x = float(line[30:38])
                    y = float(line[38:46])
                    z = float(line[46:54])
                    element = line[77:79].strip() if len(line) > 77 else 'X'
                    if element not in HYDROGEN_TYPES:
                        coords.append([x, y, z])
                except:
                    pass
            elif line.startswith('ENDMDL'):
                break
    return np.array(coords) if coords else np.array([])


def find_contacts(pose_pdbqt, receptor_pdbqt, cutoff=CONTACT_DISTANCE):
    """
    Find receptor residues within cutoff distance of the docked ligand pose.
    Reads both files as PDBQT -- no separate PDB file needed.
    Returns a list of (residue_name, residue_number, chain, min_distance) tuples.
    """
    ligand_coords = parse_pdbqt_coords(pose_pdbqt)
    if len(ligand_coords) == 0:
        return []

    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('receptor', receptor_pdbqt)

    contacts = []
    seen_residues = set()

    for model in structure:
        for chain in model:
            for residue in chain:
                resname = residue.get_resname()
                resnum = residue.id[1]
                chain_id = chain.id
                res_key = (resname, resnum, chain_id)

                if res_key in seen_residues:
                    continue

                for atom in residue:
                    # PDBQT's AutoDock atom-type column doesn't line up with
                    # the standard PDB element column, so Bio.PDB's atom.element
                    # can't be trusted here (e.g. it reads heme Fe as "F").
                    # The atom name column is unaffected by that shift.
                    if atom.get_name().strip().upper().startswith('H'):
                        continue
                    atom_coord = atom.get_coord()
                    diffs = ligand_coords - atom_coord
                    distances = np.sqrt((diffs**2).sum(axis=1))
                    min_dist = distances.min()

                    if min_dist <= cutoff:
                        contacts.append((
                            resname,
                            resnum,
                            chain_id,
                            round(float(min_dist), 3)
                        ))
                        seen_residues.add(res_key)
                        break
        break

    contacts.sort(key=lambda x: x[3])
    return contacts


print('Contact analysis function defined.')
print(f'Contact cutoff: {CONTACT_DISTANCE} Angstroms')
print('Contacts will be extracted directly from PDBQT files.')

Contact analysis function defined.
Contact cutoff: 4.5 Angstroms
Contacts will be extracted directly from PDBQT files.


### Cell 7 -- Run the docking pipeline

Docks all pending ligands against the selected target. Updates the database after each molecule. Safe to interrupt and restart -- Cell 5 will pick up from where this stopped.

Expected runtime: approximately 1-2 minutes per molecule at exhaustiveness 8. Plan for overnight for the full library.

In [ ]:
def run_docking_pipeline(pending_ligands, docking_config, selected, db_path):
    """
    Dock all pending ligands against the selected target.
    Records results and contacts in the database after each molecule.
    """
    if not pending_ligands:
        print('No molecules to dock. All done for this target.')
        return
    
    target_name = docking_config['target_name']
    target_info = docking_config['target_info']
    grid = target_info['grid_box']
    receptor_pdbqt = target_info['receptor_pdbqt']
    poses_dir = Path(selected['poses_dir']) / target_name
    poses_dir.mkdir(parents=True, exist_ok=True)
    
    total = len(pending_ligands)
    success_count = 0
    failed_count = 0
    import time
    start_time = time.time()
    
    # Progress widgets
    progress = widgets.IntProgress(
        value=0, min=0, max=total,
        description='Docking:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='700px')
    )
    status_label = widgets.Label(value='Initializing Vina...')
    display(progress)
    display(status_label)
    
    # Initialize Vina once -- much faster than reinitializing per molecule
    v = Vina(sf_name='vina', cpu=docking_config['cpu'], verbosity=0)
    v.set_receptor(receptor_pdbqt)
    v.compute_vina_maps(
        center=[grid['center_x'], grid['center_y'], grid['center_z']],
        box_size=[grid['size_x'], grid['size_y'], grid['size_z']]
    )
    
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    last_affinity = 0.0
    
    for i, (chembl_id, name, ligand_pdbqt) in enumerate(pending_ligands):
        try:
            # Set ligand
            v.set_ligand_from_file(ligand_pdbqt)
            
            # Dock
            v.dock(
                exhaustiveness=docking_config['exhaustiveness'],
                n_poses=docking_config['n_poses']
            )
            
            # Get energies
            energies = v.energies(n_poses=docking_config['n_poses'])
            affinities = [e[0] for e in energies]
            
            affinity_best = affinities[0] if len(affinities) > 0 else None
            affinity_2nd  = affinities[1] if len(affinities) > 1 else None
            affinity_3rd  = affinities[2] if len(affinities) > 2 else None
            
            if affinity_best is not None:
                last_affinity = affinity_best
            
            # Save best pose
            pose_file = str(poses_dir / f'{chembl_id}_best_pose.pdbqt')
            v.write_poses(pose_file, n_poses=1, overwrite=True)
            
            # Insert docking result
            cursor.execute('''
                INSERT OR IGNORE INTO docking_results
                (chembl_id, target_name, pdb_id, affinity_best, affinity_2nd, affinity_3rd,
                 n_poses, pose_file, exhaustiveness,
                 grid_center_x, grid_center_y, grid_center_z,
                 grid_size_x, grid_size_y, grid_size_z,
                 status, timestamp)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            ''', (
                chembl_id, target_name, target_info['pdb_id'],
                affinity_best, affinity_2nd, affinity_3rd,
                len(affinities), pose_file,
                docking_config['exhaustiveness'],
                grid['center_x'], grid['center_y'], grid['center_z'],
                grid['size_x'], grid['size_y'], grid['size_z'],
                'success', datetime.now().isoformat()
            ))
            
            result_id = cursor.lastrowid
            
            # Find and store contacts directly from PDBQT files
            if Path(pose_file).exists():
                contacts = find_contacts(pose_file, receptor_pdbqt)
                for resname, resnum, chain_id, dist in contacts:
                    cursor.execute('''
                        INSERT INTO docking_contacts
                        (result_id, chembl_id, target_name,
                         residue_name, residue_number, chain, contact_distance)
                        VALUES (?, ?, ?, ?, ?, ?, ?)
                    ''', (result_id, chembl_id, target_name,
                          resname, resnum, chain_id, dist))
            
            success_count += 1
        
        except Exception as e:
            cursor.execute('''
                INSERT OR IGNORE INTO docking_results
                (chembl_id, target_name, pdb_id, status, error_message, timestamp)
                VALUES (?, ?, ?, ?, ?, ?)
            ''', (
                chembl_id, target_name, target_info['pdb_id'],
                'failed', str(e)[:500], datetime.now().isoformat()
            ))
            failed_count += 1
        
        # Commit every 20 molecules
        if (i + 1) % 20 == 0:
            conn.commit()
        
        # Update progress
        elapsed = time.time() - start_time
        rate = (i + 1) / elapsed
        remaining = (total - i - 1) / rate if rate > 0 else 0
        progress.value = i + 1
        status_label.value = (
            f'{i+1}/{total} -- '
            f'Success: {success_count} -- '
            f'Failed: {failed_count} -- '
            f'Best so far: {last_affinity:.1f} kcal/mol -- '
            f'Remaining: {remaining/60:.1f} min'
        )
    
    conn.commit()
    conn.close()
    
    elapsed_total = time.time() - start_time
    print(f'\nDocking complete for {target_name}.')
    print(f'Total docked: {total}')
    print(f'Success: {success_count} ({100*success_count/total:.1f}%)')
    print(f'Failed: {failed_count}')
    print(f'Total time: {elapsed_total/60:.1f} minutes')


# Validate everything is ready. Note: the receptor actually used for docking
# is target_info['receptor_pdbqt'] (from the target config), not
# selected['receptor_pdbqt'] -- the latter is only relevant for the manual
# single-target entry path in Cell 2, so it is not required here.
ready = all([
    selected['db_path'],
    selected['ligand_dir'],
    selected['poses_dir'],
    docking_config['target_name'],
    docking_config['target_info']
])

if not ready:
    print('Not ready. Complete Cell 2 first.')
elif not pending_ligands:
    print('No pending ligands. Run Cell 5 first.')
else:
    print(f'Starting docking run.')
    print(f'Target: {docking_config["target_name"]}')
    print(f'Ligands to dock: {len(pending_ligands)}')
    print(f'Exhaustiveness: {docking_config["exhaustiveness"]}')
    print(f'Poses per ligand: {docking_config["n_poses"]}')
    print(f'CPU cores: {docking_config["cpu"]}')
    print('\nYou can safely interrupt at any time.')
    print('Restart by running Cell 5 then this cell again.\n')
    run_docking_pipeline(pending_ligands, docking_config, selected, selected['db_path'])

### Cell 8 -- Quick results summary

Shows the top hits for the current target ranked by binding affinity.

In [ ]:
if selected['db_path'] and docking_config['target_name']:
    conn = sqlite3.connect(selected['db_path'])
    
    target = docking_config['target_name']
    
    # Status summary
    status_summary = pd.read_sql('''
        SELECT status, COUNT(*) as count
        FROM docking_results
        WHERE target_name = ?
        GROUP BY status
    ''', conn, params=(target,))
    
    print(f'Docking status for {target}:')
    display(status_summary)
    
    # Top 20 hits joined with molecule metadata
    top_hits = pd.read_sql('''
        SELECT
            dr.chembl_id,
            m.name,
            dr.affinity_best,
            dr.affinity_2nd,
            dr.affinity_3rd,
            m.molecular_weight,
            m.alogp,
            m.qed_score,
            m.known_antifungal,
            dr.pose_file
        FROM docking_results dr
        JOIN molecules m ON dr.chembl_id = m.chembl_id
        WHERE dr.target_name = ?
        AND dr.status = 'success'
        ORDER BY dr.affinity_best ASC
        LIMIT 20
    ''', conn, params=(target,))
    
    print(f'\nTop 20 hits by binding affinity (most negative = strongest binding):')
    display(top_hits)
    
    # Known antifungals in top hits
    known = top_hits[top_hits['known_antifungal'] == 1]
    if len(known) > 0:
        print(f'\nKnown antifungals in top 20 (pipeline validation):')
        display(known[['chembl_id', 'name', 'affinity_best']])
    
    # Affinity distribution plot
    all_affinities = pd.read_sql('''
        SELECT affinity_best FROM docking_results
        WHERE target_name = ? AND status = 'success' AND affinity_best IS NOT NULL
    ''', conn, params=(target,))
    
    if len(all_affinities) > 0:
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.hist(all_affinities['affinity_best'], bins=50, color='steelblue', alpha=0.7, edgecolor='white')
        ax.axvline(x=all_affinities['affinity_best'].quantile(0.05),
                   color='coral', linestyle='--', label='Top 5% cutoff')
        ax.set_xlabel('Binding Affinity (kcal/mol)', fontsize=12)
        ax.set_ylabel('Count', fontsize=12)
        ax.set_title(f'Docking Affinity Distribution -- {target}\n{len(all_affinities)} molecules docked', fontsize=12)
        ax.legend()
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        plt.tight_layout()
        plt.show()
    
    conn.close()
else:
    print('Complete Cells 2 and 3 first.')

### Cell 9 -- Show contacts for a specific hit

Enter a ChEMBL ID to see which protein residues it contacts in its best docking pose.

In [ ]:
contact_out = widgets.Output()
btn_contacts = widgets.Button(
    description='Enter ChEMBL ID to show contacts',
    button_style='info',
    layout=widgets.Layout(width='350px')
)

def show_contacts(b):
    with contact_out:
        clear_output()
        chembl_id = ask_string(
            'ChEMBL ID',
            'Enter the ChEMBL ID of the compound to inspect:',
            initial='CHEMBL'
        )
        if not chembl_id:
            return
        
        conn = sqlite3.connect(selected['db_path'])
        
        result = pd.read_sql('''
            SELECT dr.chembl_id, m.name, dr.affinity_best, dr.target_name, dr.pdb_id
            FROM docking_results dr
            JOIN molecules m ON dr.chembl_id = m.chembl_id
            WHERE dr.chembl_id = ? AND dr.target_name = ?
        ''', conn, params=(chembl_id, docking_config['target_name']))
        
        if len(result) == 0:
            print(f'No docking result found for {chembl_id} against {docking_config["target_name"]}')
            conn.close()
            return
        
        print(f'Compound: {result.iloc[0]["name"]} ({chembl_id})')
        print(f'Target: {result.iloc[0]["target_name"]} ({result.iloc[0]["pdb_id"]})')
        print(f'Best affinity: {result.iloc[0]["affinity_best"]:.2f} kcal/mol')
        
        contacts = pd.read_sql('''
            SELECT residue_name, residue_number, chain, contact_distance
            FROM docking_contacts
            WHERE chembl_id = ? AND target_name = ?
            ORDER BY contact_distance ASC
        ''', conn, params=(chembl_id, docking_config['target_name']))
        
        conn.close()
        
        if len(contacts) > 0:
            print(f'\nContacting residues (within {CONTACT_DISTANCE} Angstroms):')
            display(contacts)
        else:
            print('No contact data found.')

btn_contacts.on_click(show_contacts)
display(btn_contacts)
display(contact_out)

### Cell 10 -- Export top hits for further analysis

Exports the top N hits with all metadata and contact information to a CSV file.

In [ ]:
export_out = widgets.Output()
btn_export = widgets.Button(
    description='Export top hits to CSV',
    button_style='success',
    layout=widgets.Layout(width='280px')
)

def export_hits(b):
    with export_out:
        clear_output()
        n = ask_integer('Top N hits', 'How many top hits to export?', 100)
        if not n:
            return
        save_dir = browse_directory('Select folder to save export')
        if not save_dir:
            return
        
        target = docking_config['target_name']
        conn = sqlite3.connect(selected['db_path'])
        
        df = pd.read_sql(f'''
            SELECT
                dr.chembl_id,
                m.name,
                m.smiles,
                dr.affinity_best,
                dr.affinity_2nd,
                dr.affinity_3rd,
                m.molecular_weight,
                m.alogp,
                m.hbd,
                m.hba,
                m.psa,
                m.qed_score,
                m.fsp3,
                m.lipinski_pass,
                m.known_antifungal,
                dr.pose_file
            FROM docking_results dr
            JOIN molecules m ON dr.chembl_id = m.chembl_id
            WHERE dr.target_name = ?
            AND dr.status = 'success'
            ORDER BY dr.affinity_best ASC
            LIMIT ?
        ''', conn, params=(target, n))
        
        conn.close()
        
        export_path = Path(save_dir) / f'{target}_top{n}_hits.csv'
        df.to_csv(export_path, index=False)
        
        print(f'Exported {len(df)} hits to: {export_path}')
        print(f'Best affinity in set: {df["affinity_best"].min():.2f} kcal/mol')
        print(f'Worst affinity in set: {df["affinity_best"].max():.2f} kcal/mol')
        display(df.head(10))

btn_export.on_click(export_hits)
display(btn_export)
display(export_out)

---
## Running against multiple targets

To dock against a different target, go back to Cell 3, click Select Target, choose a different target name, then run Cell 5 and Cell 7 again. The checkpoint system ensures each target accumulates results independently without interference.

**GitHub:** https://github.com/sardism/BasementDrugDiscovery

**Note:** All results are computational predictions requiring experimental validation.